In [2]:
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt
from fastmri.data.transforms import tensor_to_complex_np, to_tensor
import h5py
from data_utils import *
from datasets import *
from hash_encoding_batch import *

In [3]:
levels = 5
n_min = 45
n_features_per_level = 5
n_max = 320
log2_hashmap_size = 13
b = np.exp((np.log(n_max) - np.log(n_min)) / (levels - 1))


size  = 320
x = torch.arange(size)
y = torch.arange(size)
z = torch.arange(4)
coil = torch.arange(3)

points = torch.meshgrid(x, y, z, coil, indexing="ij")
points = torch.stack(points, dim=-1).reshape(-1, len(points)).float()
xy = points[:,:2]

In [5]:
from pathlib import Path
path_to_data = Path('/itet-stor/mcrespo/bmicdatasets-originals/Originals/fastMRI/brain/multicoil_train/')
n_volumes = 2
vol_id0 = 0
if path_to_data.is_dir():
        files = sorted(
            [
                file
                for file in path_to_data.iterdir()
                if file.suffix == ".h5" and "AXT1POST_205" in file.name
            ]
        )[vol_id0:vol_id0+n_volumes]
        
ground_truth = []
kspace_gt = []
for i,vol in enumerate(files):
    file = vol
    with h5py.File(file, "r") as hf:
        ground_truth.append(
            hf["reconstruction_rss"][()][: 2]
        )
        
        kspace_gt.append(to_tensor(preprocess_kspace(hf["kspace"][()][: 2])))

In [ ]:
dataset = KCoordDataset(path_to_data, n_volumes=2, n_slices=2, with_mask=True, acceleration=4, center_frac=0.15)

Training with Equispaced mask
Training with Equispaced mask


In [5]:
folder_data = '/itet-stor/mcrespo/bmicdatasets-originals/Originals/fastMRI/brain/multicoil_train/'

for file in files:
    file_data = os.path.join(folder_data, file)

    embeddings = torch.nn.Embedding(
        len(dataset.metadata), 512
    )

    coord_dim  = 3
    L = 10

    L_mult = torch.pow(2, torch.arange(L)) * np.pi
    # register_buffer("L_mult", L_mult)
    coord_encoding_dim = L * 2 * coord_dim

    x = points.unsqueeze(-1) * L_mult
    x = torch.cat([torch.sin(x), torch.cos(x)], dim=-1)
    x = x.view(x.size(0), -1)

TypeError: KCoordDataset.__init__() got an unexpected keyword argument 'center_train'

In [18]:
import pandas as pd


n_features = 5

def _get_number_of_embeddings(level_idx: int) -> int:
    max_size = 2 ** log2_hashmap_size
    n_l = int(n_min * (b ** level_idx).item())
    n_l_embeddings = (n_l + 5) ** 2
    return min(max_size, n_l_embeddings)

def bilinear_interp(x: torch.Tensor, box_indices: torch.Tensor, box_embedds: torch.Tensor) -> torch.Tensor:
    device = x.device
    
    if box_indices.shape[1] > 2:
        weights = torch.norm(box_indices - x[:, None, :], dim=2)
        den = weights.sum(dim=1, keepdim=True)
        
        weights /= den # Normalize weights
        weights = 1-weights # NOTE: More weight is given to vertex closer to the point of interest
        
        weights = weights.to(device)
        box_embedds = box_embedds.to(device)

        Npoints = len(den)
        xi_embedding = torch.zeros((Npoints, n_features), device = device)
        print(xi_embedding.shape)
        for i in range(4): # For each corner of the box
            print(box_embedds[:,i,:].shape)
            print(weights[:,i].shape)
            xi_embedding += weights[...,i].unsqueeze(1) * box_embedds[...,i,:]
            # xi_embedding += weights[:,i].unsqueeze(1) * box_embedds[:,i,:]
            
    else:
        xi_embedding = box_embedds
        
    return xi_embedding

def _get_box_idx(points: torch.Tensor, n_l: int) -> tuple:
    
    # Get bounding box indices for a batch of points
    if points.dim() > 1:
        x = points[:,0]
        y = points[:,1]
    else:
        x = points[0]
        y = points[1]

    if n_max == n_l:
        box_idx = points
        hashed_box_idx = _hash(points)
    else:
        # Calculate box size based on the total boxes
        box_width = n_max // n_l  # Width of each box
        box_height = n_max // n_l  # Height of each box

        x_min = torch.maximum(torch.zeros_like(x), (x // box_width) * box_width)
        y_min = torch.maximum(torch.zeros_like(y), (y // box_height) * box_height)
        x_max = torch.minimum(torch.full_like(x, n_max), x_min + box_width)
        y_max = torch.minimum(torch.full_like(y, n_max), y_min + box_height)
        
        # Stack to create four corners per point, maintaining the batch dimension
        box_idx = torch.stack([
            torch.stack([x_min, y_min], dim=1),
            torch.stack([x_max, y_min], dim=1),
            torch.stack([x_min, y_max], dim=1),
            torch.stack([x_max, y_max], dim=1)
        ], dim=1)  # Shape: (batch_size, 4, 2)
        
        # Determine if the coordinates can be directly mapped or need hashing
        max_hashtable_size = 2 ** log2_hashmap_size
        if max_hashtable_size >= (n_l + 5) ** 2:
            hashed_box_idx, _ = _to_1D(box_idx, n_l)
        else:
            hashed_box_idx = _hash(box_idx)
            
    return box_idx, hashed_box_idx

## Hash encoders
def _to_1D(coors, n_l):

    scale_factor = n_max // n_l
    scaled_coords = torch.div(coors, scale_factor, rounding_mode="floor").int()    
    x = scaled_coords[...,0]
    y = scaled_coords[...,1]
    
    return (y * n_l + x), scaled_coords


def _hash(coords: torch.Tensor) -> torch.Tensor:
    """
    coords: this function can process upto 7 dim coordinates
    log2T:  logarithm of T w.r.t 2
    """
    device = coords.device
    primes = torch.tensor([
        1,
        2654435761,
        805459861,
        3674653429,
        2097192037,
        1434869437,
        2165219737,
    ], dtype = torch.int64, device=device
    )

    xor_result = torch.zeros(coords.shape[:-1], dtype=torch.int64, device=device)

    for i in range(coords.shape[-1]): # Loop around all possible dimensions of the vector containing the bounding box positions
        xor_result ^= coords[...,i].to(torch.int64)*primes[i]
        
    hash_mask = (1 << log2_hashmap_size) - 1
    return xor_result & hash_mask

embeddings = nn.ModuleList([
            nn.Embedding(_get_number_of_embeddings(i), n_features_per_level)
            for i in range(levels)])

vol_embeddings = []
for vol in range(n_volumes):
    vol_embeddings.append(nn.ModuleList([
            nn.Embedding(_get_number_of_embeddings(i), n_features_per_level)
            for i in range(levels)]))


In [8]:

points = torch.tensor([
    [  1, 294, 138, 1, 11],
       [  1,  40, 110, 2, 18],
       [  1, 125, 194, 2,  7],
       [  0, 146, 191, 2, 16],
       [  0,   2, 143, 1, 19],
       [  0, 240, 141, 0,  7],
       [  1,  47, 111, 2,  6],
       [  0, 298,  16, 3, 20],
       [  1, 108, 120, 1, 19],
       [  0,  25, 225, 2, 10]], dtype = torch.float32)

coords_vID = points[:,:-1]
coords = points[:,1:-1]

xy = coords[:,:-1]

In [17]:
## Multivolume hash encodings

hash_feature_size = n_features_per_level * levels + 1
model_input = torch.zeros(coords_vID.shape[0], hash_feature_size)
for vol in range(n_volumes):
    mask_vol = (coords_vID[:,0] == vol)
    reduced_batch = coords_vID[mask_vol]
    
    xy = reduced_batch[:,1:-1]
    xy_embedded_all = []
    
    for i in range(levels):
        n_l = int(n_min * b ** i)
        
        box_indices, hashed_box_idx = _get_box_idx(xy, n_l)
        box_embedds = vol_embeddings[vol][i](hashed_box_idx)

        xy_embedded = bilinear_interp(xy, box_indices, box_embedds)
        
        xy_embedded_all.append(xy_embedded)
        
    xy_embeddings_all = torch.cat(xy_embedded_all, dim=1)
    full_embedding = torch.cat((xy_embeddings_all, reduced_batch[:,3].unsqueeze(-1)), dim=1)
    
    model_input[mask_vol] = full_embedding

torch.Size([5, 3])
torch.Size([5, 5])
torch.Size([5])


RuntimeError: The size of tensor a (3) must match the size of tensor b (5) at non-singleton dimension 1

In [82]:
for i in range(levels):
    hash_vol_index = coords_vID*i + levels

In [78]:
xy_embedded_all = []
box_embeds_appended = []
for i in range(levels):
    n_l = int(n_min * b ** i)
    
    box_indices, hashed_box_idx = _get_box_idx(xy, n_l)
    
    box_embedds = embeddings[i](hashed_box_idx)
    
    box_embeds_appended.append(box_embedds)

    xy_embedded = bilinear_interp(xy, box_indices, box_embedds)
    
    xy_embedded_all.append(xy_embedded)
    
    
xy_embedded_all = torch.cat(xy_embedded_all, dim = 1)    


torch.Size([10, 3])
torch.Size([10])
torch.Size([10, 3])
torch.Size([10])
torch.Size([10, 3])
torch.Size([10])
torch.Size([10, 3])
torch.Size([10])
torch.Size([10, 3])
torch.Size([10])
torch.Size([10, 3])
torch.Size([10])
torch.Size([10, 3])
torch.Size([10])
torch.Size([10, 3])
torch.Size([10])


In [61]:
box_embeds_appended[0].shape

torch.Size([10, 4, 3])